### BayFlux Demo Notebook: Sample Toya 2010 WT5h data with E. coli core model
By Tyler W. H. Backman

In this notebook we import data from the following four files, created in an accompanying notebook:
* **File 1:** Metabolic model (.sbml format)
* **File 2:** Extracellular exchange and flux bounds (.csv format)
* **File 3:** Atom transitions for each reaction (.txt format)
* **File 4:** Mass distribution experimental data (.tab format)

#### Load Python libraries

In [ ]:
%matplotlib inline

In [ ]:
import os
import cobra
from cobra.sampling import MCMCACHRSampler
import bayflux
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
print(np.__version__)
print(cobra.__version__)
print(bayflux.__version__)

#### Filenames for data import

In [ ]:
# Data directory
dataDir = 'ecoli_core_model'
outputDataDir = os.path.join(dataDir, 'outputdata')
# Ensure both directories exist
os.makedirs(outputDataDir, exist_ok=True)

# File 1: Metabolic model (.sbml format)
modelFile = os.path.join(dataDir, 'TOYA.xml')

# File 2: Extracellular exchange and flux bounds (.csv format)
fluxBoundsFile = os.path.join(dataDir, 'EXCHANGE_FLUXES_TOYA2.csv')

# File 3: Atom transitions for each reaction (.txt format)
transitionsFile = os.path.join(dataDir, 'TOYA_transitions.txt')

# File 4: Mass distribution experimental data (.tab format)
mdvFile = os.path.join(dataDir, 'TOYA_wt5h_Mass_Distibution.txt')

#### Import model and set flux bounds

In [ ]:
# Read the model file
cobrapymodel = cobra.io.read_sbml_model(modelFile)
model = bayflux.ReactionNetwork(cobrapymodel)
model

Import and apply exchange fluxes from file

In [ ]:
# count the number of reactions we have transitions for
sum([len(r.transitions) > 0 for r in model.reactions])

In [ ]:
model.readFluxConstraints(fluxBoundsFile)

Set some flux bounds to limit the search space of the model. For test purposes we have narrow bounds here
so the notebook runs quickly, for real world models much larger bounds may be necessary and reasonable.

In [ ]:
# set maximum flux to limit the search space
limit = 58.5
for reaction in model.reactions:
    if reaction.lower_bound < -limit:
        reaction.lower_bound = -limit
    if reaction.upper_bound > limit:
        reaction.upper_bound = limit

Run FBA (Flux Balance Analysis) to confirm that the model is valid and can carry flux

In [ ]:
model.optimize()

#### Read atom transitions and apply to model

In [ ]:
model.readAtomTransitions(transitionsFile)

#### Read mass distribution data

In [ ]:
importedMDVs = bayflux.readMassDistribution(model, mdvFile)
importedMDVs

#### Specify the number of extra neutrons in each position of the carbon inputs

In [ ]:
substrateLabelingDict = {
    model.metabolites.get_by_id('glc_D_e'): ((0.3, [1,0,0,0,0,0]),(0.2, [1,1,1,1,1,1]),(0.5, [0,0,0,0,0,0]),),
}
substrateLabelingDict

#### Pre-compile EMU network

In [ ]:
compiledData = bayflux.emuCompile(list(importedMDVs.data.keys()), model, substrateLabelingDict)

#### Placeholder: These functions will be moved into the main package before final release

In [ ]:
def normpdf(x, mean, sd):
    # based on this but with np.log added https://codereview.stackexchange.com/a/98891
    # much faster, bur same answer as norm.logpdf
    var = float(sd)**2
    denom = (2*np.pi*var)**.5
    num = math.exp(-(float(x)-float(mean))**2/(2*var))
    return np.log(num/denom)

def logLikelihood(fluxVector, lcmsResults=importedMDVs, compiledData=compiledData, substrateLabelingDict=substrateLabelingDict, errorSD=0.007041666666666667):

    # simulate labeling distribution
    try:
        results = bayflux.simulateLabeling(compiledData, fluxVector, substrateLabelingDict)
    except np.linalg.LinAlgError:
        print('LinAlgError')
        return np.finfo(np.float32).min # return a really low probability if we get an error
    
    logLikelihood = 0
    
    # find overlap between results and predicted EMUs
    for emu, mdvs in lcmsResults.data.items():
        emuHash = hash(emu)
        found = False
        for singleSize in compiledData:
            if emuHash in singleSize['matrixCoords']['internalEMUs']['hashes']:
                index = singleSize['matrixCoords']['internalEMUs']['hashes'].index(emuHash)
                found = True
                # zip together experimental results and simulated results into pairwise
                # tuples
                for mdv in mdvs:
                    for comparison in zip(mdv, results[singleSize['size']][index]):
                        logLikelihood += normpdf(comparison[0], comparison[1], errorSD)
        assert found == True, 'missing MDV in simulated results for ' + repr(emu)
    return logLikelihood

In [ ]:
def pairPlot(samples, reactions, thinning=1):
    # plot sample distribution for warmup samples
    g = sns.PairGrid(samples[reactions].iloc[::thinning, :], diag_sharey=False) # plot only every 10th sample
    # g = sns.PairGrid(samples[reactions], diag_sharey=False)
    g.map_lower(plt.hexbin, gridsize=25, cmap=plt.get_cmap('Blues'), linewidths=0)
    g.map_diag(sns.kdeplot, lw=3, gridsize=100)
    for i, j in zip(*np.triu_indices_from(g.axes, 1)):
        g.axes[i, j].set_visible(False)

#### Initialize sampler and collect uniform samples to identify the center of the metabolic model

In [ ]:
# initialize the sampler
mcmcachr = cobra.sampling.MCMCACHRSampler(model, thinning=100, seed=850)

In [ ]:
%%time
# should take 2-3 minutes for 1M samples
# collect uniform samples in order to identify the center of the model
warmupsamples = 100000
centeringSamples = mcmcachr.sample(warmupsamples, fluxes=False, validatecheck=True)

In [ ]:
type(centeringSamples)

In [ ]:
np.save(os.path.join(outputDataDir, 'ecoli_core_center_old.npy'), mcmcachr.center, allow_pickle=False)

In [ ]:
# show sample validation (1.0 = valid)
tol = 1e-6
pd.Series([int(x) for x in (mcmcachr.validate(centeringSamples, feas_tol=tol, bounds_tol=tol) == 'v')]).plot()

In [ ]:
# create a list of the reactions we want to plot fluxes for
reactionsToPlot = ['EDD', 'ICL', 'MALS']

In [ ]:
# inspect mixing in the last  warmup samples
centeringSamples[reactionsToPlot][-500:].plot()

In [ ]:
pairPlot(centeringSamples, reactionsToPlot)

#### Infer 13C MFA Posterior distribution from 13C Data using MCMC

In [ ]:
%%time
# should take ~12m for 100k samples
posteriorsamples = 100000
posteriorSamples = mcmcachr.sample(posteriorsamples, likelihood=logLikelihood, fluxes=False, validatecheck=True)

In [ ]:
posteriorSamples.shape

In [ ]:
np.save(os.path.join(outputDataDir, 'ecoli_core_mcmc.npy'), posteriorSamples, allow_pickle=False)

In [ ]:
# show frequency of validtions
# from cobrapy code:
#            - 'v' means feasible in bounds and equality constraints
#            - 'l' means a lower bound violation
#            - 'u' means a lower bound validation
#            - 'e' means and equality constraint violation
            
validationResult = pd.Series(mcmcachr.validate(centeringSamples))
validationResult.value_counts()

In [ ]:
# show sample validation (1.0 = valid)
tol = 1e-6
pd.Series([int(x) for x in (mcmcachr.validate(posteriorSamples, feas_tol=tol, bounds_tol=tol) == 'v')]).plot()

In [ ]:
# realSamples[allRxns].plot()
posteriorSamples[reactionsToPlot].plot()

In [ ]:
pairPlot(posteriorSamples, reactionsToPlot)

### Plot marginal distributions for each reaction

Here we plot a marginal distribution for each reaction as a net flux by adding the forward and reverse fluxes to get net flux for reversible reactions.


In [ ]:
centeringSamples_df = pd.DataFrame(centeringSamples)
centeringSamples_df

In [ ]:
posteriorSamples_df = pd.DataFrame(posteriorSamples)
# Set column names of posteriorSamples_df to match those of centeringSamples_df
posteriorSamples_df.columns = centeringSamples_df.columns
posteriorSamples_df

In [ ]:
# summarize the samples to be plotted

deleteStart = 0
netSamples = np.subtract(posteriorSamples_df.iloc[deleteStart::1,0::2], posteriorSamples_df.iloc[deleteStart::1,1::2])
netUniform = np.subtract(centeringSamples_df.iloc[deleteStart::1,0::2], centeringSamples_df.iloc[deleteStart::1,1::2])
display(netUniform.shape)
display(netSamples.shape)

In [ ]:
# remove reactions with no variance, e.g. they cannot carry flux or were fully constrained

rxnsWithVariance = netUniform.std(axis=0) > 0.001
netUniform = netUniform.loc[:,rxnsWithVariance]
netSamples = netSamples.loc[:,rxnsWithVariance]

In [ ]:
display(netUniform.shape)
display(netSamples.shape)

In [ ]:
warmupSampleThinning = math.ceil(centeringSamples_df.shape[0]/warmupsamples)
warmupSampleThinning

In [ ]:
realSampleThinning = math.ceil(posteriorSamples_df.shape[0]/posteriorSamples_df.shape[0])
realSampleThinning

In [ ]:
import matplotlib
# plot marginals

cols = 3
rows = math.ceil(netUniform.shape[1]/cols)

matplotlib.rcParams['figure.figsize'] = (16,3)
for j in range(0, rows):
    for k in range(0, cols):
        i = (j*cols) + k
        if i > (netUniform.shape[1] - 1):
            break
        # try:
        plt.subplot(1, cols, k + 1)
        sns.kdeplot(netUniform.iloc[0::warmupSampleThinning,i], lw=2, gridsize=100, color="grey", label="Uniform")
        sns.kdeplot(netSamples.iloc[0::realSampleThinning,i], lw=4, gridsize=100, color="darkred")
        # except:
            # continue

        plt.xlabel(netUniform.columns[i] + " Flux")
        if k == 0:
            plt.ylabel("Probability density")
    plt.show()

# Now comparing files from MP-MCMC runs
### MCMC vs MP-MCMC

In [ ]:
# Load the result arrays from the specified file paths
netSamples_old = np.load('/bayflux/Test-5/2B-ecoli_core_model/MP-MCMC/4chains_120M-powerIter_run2_modTranProb/outputdata/job1_bayflux_ecoli_genomescale001.npy')
print(f"shape of the combined array from old dataset: {netSamples_old.shape}")

# set column names as reaction ids
reaction_ids = [reaction.id for reaction in model.reactions]
print(f"number of reactions present in the model: {len(reaction_ids)}")

# Perform the subtraction operation to get the net fluxes
netSamples_old_diff = netSamples_old[:, ::2] - netSamples_old[:, 1::2]
netSamples_oldDF = pd.DataFrame(netSamples_old_diff, columns=reaction_ids)
print(f"shape of the combined array from old dataset with net fluxes: {netSamples_oldDF.shape}")

# keep only those reactions have variance i,e. can carry flux
netSamples_oldDF = netSamples_oldDF.loc[:,rxnsWithVariance]
print(f"shape of the combined array from old dataset with net fluxes and reaction with variance: {netSamples_oldDF.shape}")

In [ ]:
# Plot marginals
cols = 3
rows = math.ceil(netUniform.shape[1] / cols)


matplotlib.rcParams['figure.figsize'] = (16, 3)
for j in range(0, rows):
    plt.figure()  # Start a new figure for each row
    for k in range(0, cols):
        i = (j * cols) + k
        if i > (netUniform.shape[1] - 1):
            break
        
        plt.subplot(1, cols, k + 1)
        # warmup samples
        #sns.kdeplot(pd.Series(netUniform_old[0::warmupSampleThinning, i]), lw=1, gridsize=100, color="g", label="old-warm" if k == 0 else "")
        #sns.kdeplot(netUniform.iloc[0::warmupSampleThinning,i], ls='--', lw=2, gridsize=100, color="y", label="new-warmup" if k == 0 else "")
        
        # bayflux generated real samples
        sns.kdeplot(netSamples_oldDF.iloc[0::realSampleThinning, i], lw=1, gridsize=100, color="b", label="MP-MCMC")
        sns.kdeplot(netSamples.iloc[0::realSampleThinning, i], lw=1, ls='--', gridsize=100, color="darkred", label="MCMC")
        
        plt.xlabel(netUniform.columns[i] + " Flux")
        
        if k == 0:  # Show legend only in the first plot of each row
            plt.ylabel("Probability density")
        plt.legend(frameon=False)
    
    plt.subplots_adjust(wspace=0.18)  # Adjust space between plots
    plt.show()

#### Now calculate the convergence with four independent chains using MCMC

In [ ]:
### calculate 4 independent chains

warmupsamples = 100000
posteriorsamples = 100000
thinning = 100

#### Run 4 jobs serially one at a time

In [ ]:
%%time

posteriorSampleArray = []

# List of predefined seeds
seeds = [850, 3256, 15, 9672]

for chain in range(4):
    print(f"=> computing data for chain-{chain+1}")
    
    # Use the predefined seed for the current chain
    seed = seeds[chain]
    print(f"seed: {seed}")
    mcmcachr = cobra.sampling.MCMCACHRSampler(model, thinning=100, seed=seed)
    
    # Collect uniform samples in order to identify the center of the model
    centeringSamples = mcmcachr.sample(warmupsamples, fluxes=False, validatecheck=True)
    
    posteriorSamples = mcmcachr.sample(posteriorsamples, fluxes=False, likelihood=logLikelihood, validatecheck=True)
    posteriorSampleArray.append(posteriorSamples)
    
    # Save the posterior samples for this chain
    filename = os.path.join(outputDataDir, f'job{chain+1}_output.npy')
    np.save(filename, posteriorSamples, allow_pickle=False)
    print(f"Posterior samples for chain {chain+1} saved to {filename}")

#### Run 4 jobs simulteniously

In [ ]:
### calculate 4 independent chains parallely mpi like

warmupsamples = 100000
posteriorsamples = 100000
thinning = 100

In [ ]:
%%time

import concurrent.futures
import os
import numpy as np
import cobra

# Function to run a single chain in parallel
def run_parallel_chain(chain, seed, model, warmup_samples, posterior_samples, log_likelihood, output_data_dir):
    print(f"=> computing data for parallel chain-{chain+1}")
    print(f"seed: {seed}")
    mcmcachr = cobra.sampling.MCMCACHRSampler(model, thinning=100, seed=seed)
    
    # Collect uniform samples in order to identify the center of the model
    centering_samples = mcmcachr.sample(warmupsamples, fluxes=False, validatecheck=True)
    
    posterior_samples = mcmcachr.sample(posteriorsamples, fluxes=False, likelihood=log_likelihood, validatecheck=True)
    
    # Save the posterior samples for this chain
    filename = os.path.join(output_data_dir, f'parallel_job{chain+1}_output.npy')
    np.save(filename, posterior_samples, allow_pickle=False)
    print(f"Posterior samples for parallel chain {chain+1} saved to {filename}")
    
    return posterior_samples

# List of predefined seeds for parallel execution
parallel_seeds = [850, 3256, 15, 9672]

# Placeholder for posterior samples from parallel execution
parallel_posterior_sample_array = []

# Run chains in parallel
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    parallel_futures = []
    for chain in range(4):
        seed = parallel_seeds[chain]
        parallel_futures.append(executor.submit(run_parallel_chain, chain, seed, model, warmup_samples, posterior_samples, log_likelihood, output_data_dir))
    
    for future in concurrent.futures.as_completed(parallel_futures):
        parallel_posterior_sample_array.append(future.result())

# Optional: Print or process the results from parallel execution
print("All parallel chains completed.")

#### Output analysis

In [ ]:
realSamples = np.concatenate(posteriorSampleArray)
realSamples.shape

In [ ]:
data = np.concatenate(posteriorSampleArray)
#warmupsamples = 100000
#posteriorsamples = 100000
# Print the shape of the combined data
print(f"Shape of combined data: {data.shape}")

np.save(os.path.join(outputDataDir, 'ecoli_core_old_4chains.npy'), data, allow_pickle=False)
# compute the shape of data
chains = int(data.shape[0] / posteriorsamples)
print('found ' + str(chains) + ' markov chains!')

In [ ]:
outputDataDir

In [ ]:
%%time

warmupSamples = mcmcachr.sample(warmupsamples, fluxes=False, likelihood=False)
warmupSamples.shape

In [ ]:
realSamples = pd.DataFrame(data, columns=[v.name for v in model.variables])
realSamples.shape

In [ ]:
realSampleThinning = math.ceil(realSamples.shape[0]/(posteriorsamples*chains))
realSampleThinning

In [ ]:
allRxns = ['EDD', 'ICL', 'MALS']
allRxns

In [ ]:
realSamples[allRxns].iloc[0::realSampleThinning, :].plot()

In [ ]:
# plot sample distribution for samples
start = int((realSamples.shape[0]/10))
g = sns.PairGrid(realSamples[allRxns].iloc[start::realSampleThinning, :], diag_sharey=False) # plot only every 10th sample starting after sample 10k
# g = sns.PairGrid(realSamples[allRxns], diag_sharey=False)
g.map_lower(plt.hexbin, gridsize=25, cmap=plt.get_cmap('Blues'), linewidths=0)
g.map_diag(sns.kdeplot, lw=3, gridsize=100)
for i, j in zip(*np.triu_indices_from(g.axes, 1)):
    g.axes[i, j].set_visible(False)

### Plot marginal distributions for each reaction

Here we plot a marginal distribution for each reaction as a net flux, while excluding the first 10% of the samples, and by adding the forward and reverse fluxes to get net flux for reversible reactions.

In [ ]:
# summarize the samples to be plotted

deleteStart = 0
netSamples = np.subtract(realSamples.iloc[deleteStart::1,0::2], realSamples.iloc[deleteStart::1,1::2])
netUniform = np.subtract(warmupSamples.iloc[deleteStart::1,0::2], warmupSamples.iloc[deleteStart::1,1::2])
display(netUniform.shape)
display(netSamples.shape)

In [ ]:
# remove reactions with no variance, e.g. they cannot carry flux or were fully constrained

rxnsWithVariance = netUniform.std(axis=0) > 0.001
netUniform = netUniform.loc[:,rxnsWithVariance]
netSamples = netSamples.loc[:,rxnsWithVariance]

In [ ]:
display(netUniform.shape)
display(netSamples.shape)

In [ ]:
warmupSampleThinning = math.ceil(warmupSamples.shape[0]/warmupsamples)
warmupSampleThinning

In [ ]:
import matplotlib
# plot marginals

cols = 3
rows = math.ceil(netUniform.shape[1]/cols)

matplotlib.rcParams['figure.figsize'] = (16,3)
for j in range(0, rows):
    for k in range(0, cols):
        i = (j*cols) + k
        if i > (netUniform.shape[1] - 1):
            break
        # try:
        plt.subplot(1, cols, k + 1)
        sns.kdeplot(netUniform.iloc[0::warmupSampleThinning,i], lw=2, gridsize=100, color="grey", label="Uniform")
        sns.kdeplot(netSamples.iloc[0::realSampleThinning,i], lw=4, gridsize=100, color="darkred")
        # except:
            # continue

        plt.xlabel(netUniform.columns[i] + " Flux")
        if k == 0:
            plt.ylabel("Probability density")
    plt.show()

### Analyze sampler convergence with Gelman–Rubin diagnostic

Each chain sampling from the posterior effectively starts at a different random location from a uniform sample of the flux space because of the warmupSamples, which end at a different spot depending on the random seed. Values close to 1.0 (for example 1.1 or less) typically indicate convergence.

In [ ]:
samplesPerChain = posteriorsamples
samplesPerChain

In [ ]:
# compute stat with net fluxes only
# comment these out if we want to analyze each direction independently
numpysamples = np.array(realSamples)
data = np.subtract(numpysamples[:,0::2], numpysamples[:,1::2])
data.shape

In [ ]:
import arviz as az
import xarray as xr
#import concurrent.futures

# compute Gelman–Rubin diagnostic

# Define a number of 'burn in' samples which will be discarded, and not used to compute
# the GR diagnostic
burnin = int(samplesPerChain / 4)  # discard the first quarter samples

# Reshape data into a 3D array where each 1st dimension is a chain as used by pymc
samples = data.shape[0]
data3D = np.reshape(data, newshape=(chains, samplesPerChain, -1), order='C')
thin = int(math.ceil(samplesPerChain / 10))

# Initialize an array to store the Gelman-Rubin statistics
grStats = np.zeros(shape=(int((samplesPerChain - burnin) / thin), data3D.shape[2]), dtype=np.float64, order='C')

# Iteratively analyze convergence with stepsize thin using arviz's rhat function
for point in range(1, int((samplesPerChain - burnin) / thin) + 1):
    thisPoint = int(point * thin) + burnin
    subset = np.take(data3D, range(burnin + 1, thisPoint), axis=1)
    print(subset.shape)
    
    # Convert the subset into a DataArray and then an InferenceData object
    subset_xr = xr.DataArray(subset, dims=["chain", "draw", "variable"])
    idata = az.convert_to_inference_data(subset_xr)
    
    # Compute Gelman-Rubin diagnostic (R-hat) using arviz
    grStats[point - 1] = az.rhat(idata).to_array().values

In [ ]:
# plot the Gelman–Rubin diagnostic over the course of the simulation

matplotlib.rcParams['figure.figsize'] = (16,8)
# pd.DataFrame(grStats[100:,:]).plot(legend=False, ylim=(1,10))
pd.DataFrame(grStats).plot(legend=False, ylim=(1))

In [ ]:
# print out final convergence by reaction name
# we drop all NAs, as they are all reverse directions for non reversible reactions
GRbyVariable = pd.DataFrame(grStats[grStats.shape[0] - 1], index=[v.name for v in model.variables][0::2], columns=['GR']).dropna()
GRbyVariable[rxnsWithVariance].sort_values(by='GR')

In [ ]:
GRbyVariable.shape

In [ ]:
print(f"{(GRbyVariable['GR'] < 1.10).sum()} ({((GRbyVariable['GR'] < 1.10).sum()/GRbyVariable.shape[0])*100:.2f} %)")

In [ ]:
print(f"{(GRbyVariable['GR'] > 1.10).sum()} ({((GRbyVariable['GR'] > 1.10).sum()/GRbyVariable.shape[0])*100:.2f} %)")

In [ ]:
print(f"{(GRbyVariable[rxnsWithVariance] < 1.10).sum()}")

In [ ]:
print(f"{(GRbyVariable[rxnsWithVariance] > 1.10).sum()}")

## Output analysis directly from saved files

In [ ]:
# Initialize an empty list to hold the posterior samples
posteriorSampleArray = []

# Load the posterior samples from the four different .npy files
for chain in range(4):
    filename = os.path.join(outputDataDir, f'job{chain+1}_output.npy')
    posteriorSamples = np.load(filename, allow_pickle=False)
    posteriorSampleArray.append(posteriorSamples)

# Concatenate the loaded posterior samples
data = np.concatenate(posteriorSampleArray)

# Print the shape of the combined data
print(f"Shape of combined data: {data.shape}")

# Compute the number of chains
chains = int(data.shape[0] / posteriorsamples)
print('found ' + str(chains) + ' markov chains!')

In [ ]:
%%time

warmupSamples = mcmcachr.sample(warmupsamples, fluxes=False, likelihood=False)
warmupSamples.shape

In [ ]:
realSamples = pd.DataFrame(data, columns=[v.name for v in model.variables])
realSamples.shape

In [ ]:
realSampleThinning = math.ceil(realSamples.shape[0]/(posteriorsamples*chains))
realSampleThinning

In [ ]:
allRxns = ['EDD', 'ICL', 'MALS']
allRxns

In [ ]:
realSamples[allRxns].iloc[0::realSampleThinning, :].plot()

In [ ]:
# plot sample distribution for samples
start = int((realSamples.shape[0]/10))
g = sns.PairGrid(realSamples[allRxns].iloc[start::realSampleThinning, :], diag_sharey=False) # plot only every 10th sample starting after sample 10k
# g = sns.PairGrid(realSamples[allRxns], diag_sharey=False)
g.map_lower(plt.hexbin, gridsize=25, cmap=plt.get_cmap('Blues'), linewidths=0)
g.map_diag(sns.kdeplot, lw=3, gridsize=100)
for i, j in zip(*np.triu_indices_from(g.axes, 1)):
    g.axes[i, j].set_visible(False)

In [ ]:
# summarize the samples to be plotted

deleteStart = 0
netSamples = np.subtract(realSamples.iloc[deleteStart::1,0::2], realSamples.iloc[deleteStart::1,1::2])
netUniform = np.subtract(warmupSamples.iloc[deleteStart::1,0::2], warmupSamples.iloc[deleteStart::1,1::2])
display(netUniform.shape)
display(netSamples.shape)

In [ ]:
# remove reactions with no variance, e.g. they cannot carry flux or were fully constrained

rxnsWithVariance = netUniform.std(axis=0) > 0.001
netUniform = netUniform.loc[:,rxnsWithVariance]
netSamples = netSamples.loc[:,rxnsWithVariance]

In [ ]:
display(netUniform.shape)
display(netSamples.shape)

In [ ]:
warmupSampleThinning = math.ceil(warmupSamples.shape[0]/warmupsamples)
warmupSampleThinning

In [ ]:
import matplotlib
# plot marginals

cols = 3
rows = math.ceil(netUniform.shape[1]/cols)

matplotlib.rcParams['figure.figsize'] = (16,3)
for j in range(0, rows):
    for k in range(0, cols):
        i = (j*cols) + k
        if i > (netUniform.shape[1] - 1):
            break
        # try:
        plt.subplot(1, cols, k + 1)
        sns.kdeplot(netUniform.iloc[0::warmupSampleThinning,i], lw=2, gridsize=100, color="grey", label="Uniform")
        sns.kdeplot(netSamples.iloc[0::realSampleThinning,i], lw=4, gridsize=100, color="darkred")
        # except:
            # continue

        plt.xlabel(netUniform.columns[i] + " Flux")
        if k == 0:
            plt.ylabel("Probability density")
    plt.show()

In [ ]:
samplesPerChain = posteriorsamples
samplesPerChain

In [ ]:
# compute stat with net fluxes only
# comment these out if we want to analyze each direction independently
numpysamples = np.array(realSamples)
data = np.subtract(numpysamples[:,0::2], numpysamples[:,1::2])
data.shape

In [ ]:
import arviz as az
import xarray as xr
#import concurrent.futures

# compute Gelman–Rubin diagnostic

# Define a number of 'burn in' samples which will be discarded, and not used to compute
# the GR diagnostic
burnin = int(samplesPerChain / 4)  # discard the first quarter samples

# Reshape data into a 3D array where each 1st dimension is a chain as used by pymc
samples = data.shape[0]
data3D = np.reshape(data, newshape=(chains, samplesPerChain, -1), order='C')
thin = int(math.ceil(samplesPerChain / 10))

# Initialize an array to store the Gelman-Rubin statistics
grStats = np.zeros(shape=(int((samplesPerChain - burnin) / thin), data3D.shape[2]), dtype=np.float64, order='C')

# Iteratively analyze convergence with stepsize thin using arviz's rhat function
for point in range(1, int((samplesPerChain - burnin) / thin) + 1):
    thisPoint = int(point * thin) + burnin
    subset = np.take(data3D, range(burnin + 1, thisPoint), axis=1)
    print(subset.shape)
    
    # Convert the subset into a DataArray and then an InferenceData object
    subset_xr = xr.DataArray(subset, dims=["chain", "draw", "variable"])
    idata = az.convert_to_inference_data(subset_xr)
    
    # Compute Gelman-Rubin diagnostic (R-hat) using arviz
    grStats[point - 1] = az.rhat(idata).to_array().values

In [ ]:
# plot the Gelman–Rubin diagnostic over the course of the simulation

matplotlib.rcParams['figure.figsize'] = (16,8)
# pd.DataFrame(grStats[100:,:]).plot(legend=False, ylim=(1,10))
pd.DataFrame(grStats).plot(legend=False, ylim=(1))

In [ ]:
# print out final convergence by reaction name
# we drop all NAs, as they are all reverse directions for non reversible reactions
GRbyVariable = pd.DataFrame(grStats[grStats.shape[0] - 1], index=[v.name for v in model.variables][0::2], columns=['GR']).dropna()
GRbyVariable[rxnsWithVariance].sort_values(by='GR')

In [ ]:
print(f"{(GRbyVariable['GR'] < 1.10).sum()} ({((GRbyVariable['GR'] < 1.10).sum()/GRbyVariable.shape[0])*100:.2f} %)")

#### Infer 13C MFA Posterior distribution from 13C Data using Multiple Propposal MCMC (MP-MCMC)

### Map reaction names between genome scale and core model

In [ ]:
genomescalemodel = cobra.io.rac_cd_sbml_model('ecoli_genome_scale/data/ecoli_genome_scale.xml')

In [ ]:
for r in genomescalemodel.metabolites.s7p_c.reactions:
    print(r)

In [ ]:
print(genomescalemodel.reactions.TALA)

In [ ]:
genomescalemodelRxnIds = set([r.id for r in genomescalemodel.reactions])

In [ ]:
coreModelRxnIds = set([r.id for r in model.reactions])

In [ ]:
idIntersection = genomescalemodelRxnIds.intersection(coreModelRxnIds)

In [ ]:
for r in model.reactions:
    print(r)

In [ ]:
# idMap = {r.id: '' for r in model.reactions}
# for id in idIntersection:
#     idMap[id] = id

In [ ]:
idmap = {
 # '2DDA7Pbm': '',
 # '2DDA7P_exchange': '',
 # 'ACACCT': '',
 'Ac_exchange': 'EX_ac_e',
 'ACONT': 'ACONTb',
 'AKGDH': 'AKGDH',
 # 'AcCoabm': '',
 # 'AcCoa_exchange': '',
 'co2_exchange': 'EX_co2_e',
 'glc_exchange': 'EX_glc__D_e',
 'CS': 'CS',
 'DDPA': 'DDPA',
 'DHAPT': 'DHAPT',
 'EDA': 'EDA',
 'EDD': 'EDD',
 'ENO': 'ENO',
 'F6PA': 'F6PA',
 # 'F6Pbm': '',
 # 'F6P_exchange': '',
 'FBA': 'FBA',
 'FUM': 'FUM',
 'G6PDH2r': 'G6PDH2r',
 # 'G6Pbm': '',
 # 'G6P_exchange': '',
 'GAPD': 'GAPD',
 # 'GLCpts': 'GLCptspp',
 'GLUDy': 'GLUDy',
 # 'GLUbm': '',
 # 'GLU_exchange': '',
 'GND': 'GND',
 'ICDHyr': 'ICDHyr',
 'ICL': 'ICL',
 'MALS': 'MALS',
 'MDH': 'MDH',
 'ME1': 'ME1',
 # 'OAAbm': '',
 # 'OAA_exchange': '',
 'PDH': 'PDH',
 # 'PEPbm': '',
 # 'PEP_exchange': '',
 'PFK': 'PFK',
 # '3PG_exchange': '',
 'PGI': 'PGI',
 'PGK': 'PGK',
 'PGL': 'PGL',
 'PGM': 'PGM',
 'PPC': 'PPC',
 'PPCK': 'PPCK',
 'PYK': 'PYK',
 # 'PYRbm': '',
 # 'PYR_exchange': '',
 # 'R5Pbm': '',
 # 'R5P_exchange': '',
 'RPE': 'RPE',
 'RPI': 'RPI',
 'SUCD1i': 'SUCDi',
 'SUCOAS': 'SUCOAS',
 # 'TA1': '',
 # 'TA2': '',
 # 'TK1': '',
 # 'TK2': '',
 # 'TK3': '',
 'TPI': 'TPI'
}

### import genome scale fluxes

In [ ]:
# code from bayflux.subsample, because this server has an old version of bayflux without this

def subSample(inputFile, chains, warmupLength, sampleSize, model, netfluxOnly=True):
    # Reads in a set of BayFlux samples from a .npy inputFile, with a given number of chains
    # and warmup samples. Will keep a representative set of integer sampleSize samples occuring after
    # the first 'warmupLength' samples on each chain, and discard the rest. If netfluxOnly=True,
    # only net flux values are returned.
    
    # read in data
    data = np.load(inputFile, mmap_mode='r')
    
    # confirm we have enough valid samples
    assert data.shape[0] >= (sampleSize + (chains * warmupLength)), 'not enough samples'
    
    # Create a list of integers selecting 100k points to remove from the above samples
    # Discard the first 1/4 from each chain
    samplesPerChain = int(data.shape[0] / chains)

    # step size to keep
    # here we use a floating point step size, which gets converted to an integer
    # before indexing, allowing for unevenly spaced increments- thereby letting the
    # subset scale to the desired size
    stepSize = (samplesPerChain - warmupLength) / (sampleSize / chains)
    
    # store inidces samples to be kept in a list
    samplesToKeep = []

    # interate over chains and get indices of samples to keep
    for chain in range(0, chains):
              
        # create series of sample numbers to keep for this chain
        subsetToKeepThisChain = [int(x) for x in np.arange(warmupLength, samplesPerChain, stepSize)]
        
        # assert that each index is unique
        assert len(set(subsetToKeepThisChain)) == len(subsetToKeepThisChain), 'sample indices not unique- not enough samples?'
        
        # adjust sample position based on which chain we're on
        subsetToKeep = [i + (samplesPerChain * chain) for i in subsetToKeepThisChain]
        
        # append indices to list of samples to be kept
        samplesToKeep = samplesToKeep + subsetToKeep
    
    # extract the subset of samples to keep
    dataSubset = data[samplesToKeep]
    
    # put the samples in a Pandas DF
    dataSubset = pd.DataFrame(dataSubset, columns=[v.name for v in model.variables])
    
    if netfluxOnly:
        # substract both directions to get netFlux
        dataSubset = np.subtract(dataSubset.iloc[0::1,0::2], dataSubset.iloc[0::1,1::2])
        
    # cofirm we did the math right and we got the right number of samples to return    
    assert dataSubset.shape[0] == sampleSize, 'error- trying to return the wrong number of samples!'
    
    return dataSubset

In [ ]:
genomeScaleSamples = subSample('ecoli_genome_scale/outputdata/bayflux_ecoli_genomescale001.npy', 4, 10000, 100000, genomescalemodel, netfluxOnly=True)

In [ ]:
genomeScaleSamples

In [ ]:
# store list of reactions which converged in samples (from analyze notebook)
converged = ['12PPDRtex',
 '12PPDRtpp',
 '3HAD100',
 '3HAD120',
 '3HAD40',
 '3HAD60',
 '3HAD80',
 '3OAR100',
 '3OAR120',
 '3OAR140',
 '3OAR40',
 '3OAR60',
 '3OAR80',
 '3OAS100',
 '3OAS120',
 '3OAS140',
 '3OAS60',
 '3OAS80',
 'A5PISO',
 'AACTOOR',
 'ABTA',
 'ABUTD',
 'ABUTt2pp',
 'ABUTtex',
 'ACALD',
 'ACCOAC',
 'ACGK',
 'ACGS',
 'ACHBS',
 'ACKr',
 'ACLS',
 'ACOATA',
 'ACODA',
 'ACOTA',
 'ACSERtex',
 'ACSERtpp',
 'ACt2rpp',
 'ACt4pp',
 'ADK1',
 'ADNK1',
 'ADSK',
 'ADSL1r',
 'ADSS',
 'AGMHE',
 'AGMT',
 'AGPR',
 'AHCYSNS',
 'AKGDH',
 'ALAR',
 'ALCD2x',
 'ALDD2x',
 'ALR2',
 'ALR4x',
 'AOBUTDs',
 'ARBTNR1',
 'ARBTNabcpp',
 'ARBTNexs',
 'ARBTNtex',
 'ARBTNtonex',
 'ARBTNtpp',
 'ARGDC',
 'ARGORNt7pp',
 'ARGSL',
 'ARGSS',
 'ARGabcpp',
 'ARGt3pp',
 'ASAD',
 'ASNN',
 'ASNS1',
 'ASNS2',
 'ASNabcpp',
 'ASNt2rpp',
 'ASPK',
 'ASPT',
 'ASPTA',
 'AST',
 'ATPS4rpp',
 'BPNT',
 'CBMKr',
 'CBPS',
 'CD2t3pp',
 'CD2tpp',
 'CITL',
 'CO2tex',
 'CO2tpp',
 'COBALT2t3pp',
 'COBALT2tpp',
 'CRNDabcpp',
 'CRNDt2rpp',
 'CRNabcpp',
 'CRNt2rpp',
 'CTBTabcpp',
 'CTBTt2rpp',
 'CYSDS',
 'CYSS',
 'CYSabc2pp',
 'CYSabcpp',
 'CYStex',
 'CYStpp',
 'CYTBD2pp',
 'CYTBDpp',
 'CYTBO3_4pp',
 'CYTK1',
 'DADA',
 'DALAt2pp',
 'DALAtex',
 'DHAD1',
 'DHAD2',
 'DHAPT',
 'DHPTDCs',
 'DM_hmfurn_c',
 'DTMPK',
 'D_LACt2pp',
 'D_LACtex',
 'EAR100x',
 'EAR100y',
 'EAR120x',
 'EAR120y',
 'EAR40x',
 'EAR40y',
 'EAR80x',
 'EAR80y',
 'EDA',
 'EDD',
 'ETOHt2rpp',
 'ETOHtex',
 'EX_12ppd__R_e',
 'EX_4abut_e',
 'EX_acser_e',
 'EX_ala__D_e',
 'EX_co2_e',
 'EX_cys__L_e',
 'EX_etoh_e',
 'EX_fe2_e',
 'EX_fe3_e',
 'EX_fum_e',
 'EX_glcn_e',
 'EX_glu__L_e',
 'EX_h2o_e',
 'EX_h2s_e',
 'EX_h_e',
 'EX_ile__L_e',
 'EX_kdo2lipid4_e',
 'EX_lac__D_e',
 'EX_leu__L_e',
 'EX_nh4_e',
 'EX_o2_e',
 'EX_pi_e',
 'EX_ptrc_e',
 'EX_pyr_e',
 'EX_ser__L_e',
 'EX_so4_e',
 'EX_succ_e',
 'EX_urea_e',
 'EX_val__L_e',
 'F6PA',
 'FADRx',
 'FADRx2',
 'FBA',
 'FBP',
 'FE2t2pp',
 'FE2t3pp',
 'FE2tex',
 'FE2tpp',
 'FE3tex',
 'FEROpp',
 'FLDR',
 'FORt2pp',
 'FORtppi',
 'FTHFD',
 'FUM',
 'FUMt2_2pp',
 'FUMt2_3pp',
 'FUMtex',
 'G1PACT',
 'G3PD2',
 'G3PT',
 'G5SD',
 'G6PDH2r',
 'G6PP',
 'GALUi',
 'GAPD',
 'GARFT',
 'GART',
 'GF6PTA',
 'GGGABADr',
 'GGGABAH',
 'GGPTRCO',
 'GGPTRCS',
 'GHMT2r',
 'GK1',
 'GLCDpp',
 'GLCNtex',
 'GLCabcpp',
 'GLCptspp',
 'GLCt2pp',
 'GLNS',
 'GLU5K',
 'GLUABUTt7pp',
 'GLUDC',
 'GLUDy',
 'GLUN',
 'GLUSy',
 'GLUabcpp',
 'GLUt2rpp',
 'GLUt4pp',
 'GLUtex',
 'GLYAT',
 'GLYCDx',
 'GLYCK2',
 'GLYCL',
 'GLYOX',
 'GLYOX3',
 'GMHEPAT',
 'GMHEPK',
 'GMHEPPA',
 'GMPR',
 'GMPS2',
 'GND',
 'GTHRDabc2pp',
 'GTHRDabcpp',
 'GUAt2pp',
 'GUAtpp',
 'H2Otex',
 'H2Otpp',
 'H2SO',
 'H2St1pp',
 'H2Stex',
 'HCO3E',
 'HCYSMT',
 'HEX1',
 'HISabcpp',
 'HISt2rpp',
 'HSDy',
 'HSK',
 'HXPRT',
 'Htex',
 'ILETA',
 'ILEabcpp',
 'ILEt2rpp',
 'ILEtex',
 'IMPD',
 'IPMD',
 'IPPMIa',
 'IPPMIb',
 'IPPS',
 'K2L4Aabcpp',
 'K2L4Atex',
 'KARA1',
 'KARA2',
 'KAS14',
 'KAS15',
 'KDOCT2',
 'KDOPP',
 'KDOPS',
 'Kt2pp',
 'Kt3pp',
 'LALDO2x',
 'LCARR',
 'LDH_D',
 'LDH_D2',
 'LEUTAi',
 'LEUabcpp',
 'LEUt2rpp',
 'LEUtex',
 'LGTHL',
 'LPADSS',
 'LYSabcpp',
 'LYSt2pp',
 'LYSt3pp',
 'MALS',
 'MCITD',
 'MCITL2',
 'MCITS',
 'MCOATA',
 'MDH',
 'MDH2',
 'MDH3',
 'ME1',
 'ME2',
 'METAT',
 'MG2t3_2pp',
 'MG2tpp',
 'MG2uabcpp',
 'MGSA',
 'MICITDr',
 'MMCD',
 'MME',
 'MMM2',
 'MN2t3pp',
 'MN2tpp',
 'MNt2pp',
 'MOAT',
 'MOAT2',
 'MTHFC',
 'MTHFD',
 'NACODA',
 'NADH10',
 'NADH16pp',
 'NADH17pp',
 'NADH5',
 'NADK',
 'NADPHQR2',
 'NADPHQR3',
 'NADPPPS',
 'NADTRHD',
 'NAt3_1p5pp',
 'NAt3_2pp',
 'NAt3pp',
 'NDPK1',
 'NDPK2',
 'NDPK3',
 'NDPK4',
 'NDPK8',
 'NH4tex',
 'NH4tpp',
 'NI2abcpp',
 'NI2t3pp',
 'NI2tpp',
 'NI2uabcpp',
 'NTP3',
 'NTP5',
 'NTPP2',
 'NTPP4',
 'NTPP7',
 'NTPP8',
 'O2tex',
 'O2tpp',
 'OAADC',
 'OBTFL',
 'OCBT',
 'OMCDC',
 'ORNDC',
 'ORNabcpp',
 'P5CR',
 'PAPSR',
 'PDH',
 'PFK',
 'PGAMT',
 'PGCD',
 'PGI',
 'PGK',
 'PGL',
 'PGMT',
 'PIt2rpp',
 'PItex',
 'PIuabcpp',
 'POX',
 'PPA',
 'PPA2',
 'PPC',
 'PPCK',
 'PPK2',
 'PPK',
 'PPM',
 'PPS',
 'PROD2',
 'PROabcpp',
 'PRPPS',
 'PSERT',
 'PSP_L',
 'PTAr',
 'PTRCORNt7pp',
 'PTRCTA',
 'PTRCabcpp',
 'PTRCt2pp',
 'PTRCtex',
 'PUNP1',
 'PUNP2',
 'PUNP6',
 'PYK',
 'PYRt2rpp',
 'PYRtex',
 'RHCCE',
 'RNDR1',
 'RNTR1c',
 'RPE',
 'RPI',
 'S7PI',
 'SADH',
 'SADT2',
 'SERAT',
 'SERD_L',
 'SERtex',
 'SGDS',
 'SGSAD',
 'SO4tex',
 'SOTA',
 'SSALx',
 'SSALy',
 'SUCCt2_3pp',
 'SUCCt3pp',
 'SUCCtex',
 'SUCFUMtpp',
 'SUCOAS',
 'SULR',
 'SULabcpp',
 'TALA',
 'TDP',
 'TDSK',
 'THD2pp',
 'THIORDXi',
 'THRA',
 'THRD',
 'THRD_L',
 'THRS',
 'THRabcpp',
 'THRt2pp',
 'TKT1',
 'TKT2',
 'TMPK',
 'TPI',
 'TRDR',
 'U23GAAT',
 'UAGAAT',
 'UAGDP',
 'UHGADA',
 'UMPK',
 'UREAtex',
 'UREAtpp',
 'USHD',
 'VALabcpp',
 'VALt2rpp',
 'VALtex',
 'XANt2pp',
 'XANtpp',
 'ZN2abcpp',
 'ZN2t3pp',
 'ZN2tpp',
 'ZNabcpp']

In [ ]:
# filter idmap to include only reactions which converged in the genome scale sampling
filteredidmap = {}

for key, value in idmap.items():
    if value in converged:
        filteredidmap[key] = value

In [ ]:
len(filteredidmap)

In [ ]:
27/2

### Plot fluxes for core model

In [ ]:
# reactionstoshow = 10
netSamples = posteriorSamples[list(filteredidmap.keys())]
#reactionsByCertainty = list(pd.DataFrame.std(netSamples, axis=0).sort_values().index)
reactionsByCertainty = list(pd.DataFrame.median(netSamples, axis=0).sort_values().index)
# reactionstoshow = len(reactionsByCertainty) - reactionstoshow
mostCertain = netSamples.loc[:,reactionsByCertainty[0:14]]
reactionsByCertainty.reverse()
leastCertain = netSamples.loc[:,reactionsByCertainty[0:13]]

In [ ]:
netSamples.to_csv('input_data/coressamples.csv')

In [ ]:
df1 = pd.melt(leastCertain, var_name='g', value_name='x')

In [ ]:
df1.insert(2, "source", "core")

In [ ]:
df2.insert(2, "source", "genome")

In [ ]:
mergedDF = pd.concat([df1, df2])

In [ ]:
mergedDF

In [ ]:
sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

# Initialize the FacetGrid object
pal = sns.cubehelix_palette(reactionstoshow, rot=-.25, light=.7)
# g = sns.FacetGrid(mergedDF, row="g", hue="source", aspect=15, height=.5, palette=pal)
g = sns.FacetGrid(mergedDF, row="g", hue="source", aspect=15, height=.5)


# Draw the densities in a few steps
g.map(sns.kdeplot, "x",
      bw_adjust=.5, clip_on=False,
     fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, "x", clip_on=False, color="w", lw=2, bw_adjust=.5)

# passing color=None to refline() uses the hue mapping
g.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)


# Define and use a simple function to label the plot in axes coordinates
def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)
    
g.map(label, "x")

# Set the subplots to overlap
g.figure.subplots_adjust(hspace=-.25)

# Remove axes details that don't play well with overlap
g.set_titles("")
g.set(yticks=[], ylabel="")
# g.set(xlim=(0,30))
g.despine(bottom=True, left=True)

In [ ]:
sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

# Initialize the FacetGrid object
pal = sns.cubehelix_palette(reactionstoshow, rot=-.25, light=.7)
g = sns.FacetGrid(mergedDF, row="g", hue="source", aspect=15, height=.5, palette=pal)

# Draw the densities in a few steps
g.map(sns.kdeplot, "x",
      bw_adjust=.5, clip_on=False,
      fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, "x", clip_on=False, color="w", lw=2, bw_adjust=.5)

# passing color=None to refline() uses the hue mapping
g.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)


# Define and use a simple function to label the plot in axes coordinates
def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)
    
g.map(label, "x")

# Set the subplots to overlap
g.figure.subplots_adjust(hspace=-.25)

# Remove axes details that don't play well with overlap
g.set_titles("")
g.set(yticks=[], ylabel="")
# g.set(xlim=(0,30))
g.despine(bottom=True, left=True)

In [ ]:
df = pd.melt(leastCertain, var_name='g', value_name='x')

sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

# Initialize the FacetGrid object
pal = sns.cubehelix_palette(reactionstoshow, rot=-.25, light=.7)
g = sns.FacetGrid(df, row="g", hue="g", aspect=15, height=.5, palette=pal)

# Draw the densities in a few steps
g.map(sns.kdeplot, "x",
      bw_adjust=.5, clip_on=False,
      fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, "x", clip_on=False, color="w", lw=2, bw_adjust=.5)

# passing color=None to refline() uses the hue mapping
g.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)


# Define and use a simple function to label the plot in axes coordinates
def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)
    
g.map(label, "x")

# Set the subplots to overlap
g.figure.subplots_adjust(hspace=-.25)

# Remove axes details that don't play well with overlap
g.set_titles("")
g.set(yticks=[], ylabel="")
# g.set(xlim=(0,30))
g.despine(bottom=True, left=True)

In [ ]:
df = pd.melt(mostCertain, var_name='g', value_name='x')

sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

# Initialize the FacetGrid object
pal = sns.cubehelix_palette(reactionstoshow, rot=-.25, light=.7)
g = sns.FacetGrid(df, row="g", hue="g", aspect=15, height=.5, palette=pal)

# Draw the densities in a few steps
g.map(sns.kdeplot, "x",
      bw_adjust=.5, clip_on=False,
      fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, "x", clip_on=False, color="w", lw=2, bw_adjust=.5)

# passing color=None to refline() uses the hue mapping
g.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)


# Define and use a simple function to label the plot in axes coordinates
def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)
    
g.map(label, "x")

# Set the subplots to overlap
g.figure.subplots_adjust(hspace=-.25)

# Remove axes details that don't play well with overlap
g.set_titles("")
g.set(yticks=[], ylabel="")
g.set(xlim=(-10,10))
g.despine(bottom=True, left=True)

### plot genome scale fluxes

In [ ]:
genomeScaleReactionsByCertainty = [filteredidmap[r] for r in reactionsByCertainty]
genomeScaleReactionsByCertainty

In [ ]:
plotSamples = genomeScaleSamples.loc[:,genomeScaleReactionsByCertainty]

In [ ]:
# plotSamples.to_pickle('input_data/genomescalesamples.pickle')
plotSamples.to_csv('input_data/genomescalesamples.csv')

In [ ]:
mostCertain = genomeScaleSamples.loc[:,genomeScaleReactionsByCertainty[0:14]]
reactionsByCertainty.reverse()
leastCertain = genomeScaleSamples.loc[:,genomeScaleReactionsByCertainty[0:13]]

In [ ]:
genomeScaleSamples.columns

In [ ]:
df2 = pd.melt(leastCertain, var_name='g', value_name='x')
df2

In [ ]:
df = pd.melt(leastCertain, var_name='g', value_name='x')

sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

# Initialize the FacetGrid object
pal = sns.cubehelix_palette(reactionstoshow, rot=-.25, light=.7)
g = sns.FacetGrid(df, row="g", hue="g", aspect=15, height=.5, palette=pal)

# Draw the densities in a few steps
g.map(sns.kdeplot, "x",
      bw_adjust=.5, clip_on=False,
      fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, "x", clip_on=False, color="w", lw=2, bw_adjust=.5)

# passing color=None to refline() uses the hue mapping
g.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)


# Define and use a simple function to label the plot in axes coordinates
def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)
    
g.map(label, "x")

# Set the subplots to overlap
g.figure.subplots_adjust(hspace=-.25)

# Remove axes details that don't play well with overlap
g.set_titles("")
g.set(yticks=[], ylabel="")
# g.set(xlim=(0,30))
g.despine(bottom=True, left=True)

In [ ]:
df = pd.melt(mostCertain, var_name='g', value_name='x')

sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

# Initialize the FacetGrid object
pal = sns.cubehelix_palette(reactionstoshow, rot=-.25, light=.7)
g = sns.FacetGrid(df, row="g", hue="g", aspect=15, height=.5, palette=pal)

# Draw the densities in a few steps
g.map(sns.kdeplot, "x",
      bw_adjust=.5, clip_on=False,
      fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, "x", clip_on=False, color="w", lw=2, bw_adjust=.5)

# passing color=None to refline() uses the hue mapping
g.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)


# Define and use a simple function to label the plot in axes coordinates
def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)
    
g.map(label, "x")

# Set the subplots to overlap
g.figure.subplots_adjust(hspace=-.25)

# Remove axes details that don't play well with overlap
g.set_titles("")
g.set(yticks=[], ylabel="")
g.set(xlim=(-10,10))
g.despine(bottom=True, left=True)